# P-value Diagnostics for RQ1 Metrics

This notebook helps explain why some `p_value` entries are high (e.g. > 0.1), and checks whether specific scenes/samples might be problematic.

What it does:

1. Load `metrics.csv`, `null_distribution_*.csv`, and activation index artifacts.
2. List high-p rows per metric/layer.
3. Compare observed values vs corresponding null distributions.
4. Identify scenes that appear unusually often in high-null tails.
5. Expose candidate sample keys / image paths for manual inspection.

In [18]:
from pathlib import Path
import numpy as np
import pandas as pd

# ---- Config ----
RUN_ID = "rq1_cka_pipeline-20260502-144611"
NULL_FILENAME = "null_distribution_20scenes_1000draws.csv"
P_THRESH = 0.1
TAILS = [0.95, 0.99]

# ---- Resolve project root ----
cwd = Path.cwd().resolve()
if (cwd / "README.md").exists():
    project_root = cwd
elif (cwd.parent / "README.md").exists():
    project_root = cwd.parent
else:
    raise FileNotFoundError("Could not find project root (README.md not found).")

run_dir = project_root / "results" / "runs" / RUN_ID
metrics_path = run_dir / "metrics" / "metrics.csv"
null_path = project_root / "results" / "runs" / "null_distributions" / RUN_ID / NULL_FILENAME

metrics = pd.read_csv(metrics_path)
null_df = pd.read_csv(null_path)

out_root = project_root / "results" / "diagnostics" / RUN_ID 
out_root.mkdir(parents=True, exist_ok=True)


def safe_metric_name(name: str) -> str:
    return str(name).replace("/", "_").replace(" ", "_")


def side_enrichment(df_all: pd.DataFrame, df_tail: pd.DataFrame) -> pd.DataFrame:
    all_side = pd.concat([
        df_all[["left_scene_type"]].rename(columns={"left_scene_type": "scene_type"}),
        df_all[["right_scene_type"]].rename(columns={"right_scene_type": "scene_type"}),
    ], ignore_index=True)
    tail_side = pd.concat([
        df_tail[["left_scene_type"]].rename(columns={"left_scene_type": "scene_type"}),
        df_tail[["right_scene_type"]].rename(columns={"right_scene_type": "scene_type"}),
    ], ignore_index=True)

    all_counts = all_side["scene_type"].value_counts().rename("all_count")
    tail_counts = tail_side["scene_type"].value_counts().rename("tail_count")

    out = pd.concat([all_counts, tail_counts], axis=1).fillna(0)
    out["all_count"] = out["all_count"].astype(int)
    out["tail_count"] = out["tail_count"].astype(int)
    out["all_rate"] = out["all_count"] / max(out["all_count"].sum(), 1)
    out["tail_rate"] = out["tail_count"] / max(out["tail_count"].sum(), 1)
    out["lift"] = out["tail_rate"] / out["all_rate"].replace(0, np.nan)
    return out.reset_index().rename(columns={"index": "scene_type"}).sort_values("lift", ascending=False)


def pair_enrichment(df_all: pd.DataFrame, df_tail: pd.DataFrame) -> pd.DataFrame:
    def sorted_pair(row):
        return tuple(sorted((row["left_scene_type"], row["right_scene_type"])))

    a = df_all.copy()
    t = df_tail.copy()
    a["scene_type_pair"] = a.apply(sorted_pair, axis=1)
    t["scene_type_pair"] = t.apply(sorted_pair, axis=1)

    all_counts = a["scene_type_pair"].value_counts().rename("all_count")
    tail_counts = t["scene_type_pair"].value_counts().rename("tail_count")

    out = pd.concat([all_counts, tail_counts], axis=1).fillna(0)
    out["all_count"] = out["all_count"].astype(int)
    out["tail_count"] = out["tail_count"].astype(int)
    out["all_rate"] = out["all_count"] / max(out["all_count"].sum(), 1)
    out["tail_rate"] = out["tail_count"] / max(out["tail_count"].sum(), 1)
    out["lift"] = out["tail_rate"] / out["all_rate"].replace(0, np.nan)
    out = out.reset_index().rename(columns={"index": "scene_type_pair"})
    out["scene_type_left"] = out["scene_type_pair"].map(lambda x: x[0])
    out["scene_type_right"] = out["scene_type_pair"].map(lambda x: x[1])
    return out.sort_values("lift", ascending=False)


summary_rows = []
for metric_name in sorted(null_df["metric"].dropna().unique()):
    mslug = safe_metric_name(metric_name)
    mdir = out_root / mslug
    mdir.mkdir(parents=True, exist_ok=True)

    dnull = null_df[null_df["metric"] == metric_name].copy()
    dmet = metrics[metrics["metric"] == metric_name].copy()

    files_written = []

    # All null rows
    p = mdir / f"{mslug}_null_all.csv"
    dnull.to_csv(p, index=False)
    files_written.append(p)

    # Metrics rows + high p rows
    if not dmet.empty:
        p = mdir / f"{mslug}_metrics_with_pvalues.csv"
        dmet.sort_values("p_value", ascending=False).to_csv(p, index=False)
        files_written.append(p)

        p = mdir / f"{mslug}_high_p_rows_gt_{str(P_THRESH).replace('.', '_')}.csv"
        dmet[dmet["p_value"] > P_THRESH].sort_values("p_value", ascending=False).to_csv(p, index=False)
        files_written.append(p)

    # Tail exports and enrichments
    for q in TAILS:
        qn = int(q * 100)
        thr = float(dnull["null_value"].quantile(q))
        tail = dnull[dnull["null_value"] >= thr].copy()

        p = mdir / f"{mslug}_null_q{qn}_tail.csv"
        tail.to_csv(p, index=False)
        files_written.append(p)

        p = mdir / f"{mslug}_scene_type_enrichment_q{qn}.csv"
        side_enrichment(dnull, tail).to_csv(p, index=False)
        files_written.append(p)

        p = mdir / f"{mslug}_scene_type_pair_enrichment_q{qn}.csv"
        pair_enrichment(dnull, tail).to_csv(p, index=False)
        files_written.append(p)

    summary_rows.append({
        "metric": metric_name,
        "null_rows": len(dnull),
        "metrics_rows": len(dmet),
        "files_written": len(files_written),
        "folder": str(mdir),
    })

summary = pd.DataFrame(summary_rows).sort_values("metric")
print("Export complete.")
print("Output root:", out_root)
display(summary)

Export complete.
Output root: /Users/gattimartina/Documents/EPFL/Master/MA4/CS-503-Visual-Intelligence/one-trunk-or-many/results/diagnostics/rq1_cka_pipeline-20260502-144611


,metric,null_rows,metrics_rows,files_written,folder
0,cka,12000,12,9,/Users/gattimartina/Documents/EPFL/Master/MA4/...
1,knn_overlap,12000,12,9,/Users/gattimartina/Documents/EPFL/Master/MA4/...
2,pwcca,12000,12,9,/Users/gattimartina/Documents/EPFL/Master/MA4/...
